# Séance 1 — Du LLM à l'agent outillé
### Cours « Agentic AI » — M2 I2A

**Auteur :** Mehdi Ammi — Université Paris 8, laboratoire LIASD<br>
**Conception :** TP conçu avec l'assistance de Claude Code (Anthropic)<br>
**Version :** octobre 2026

---

> **🎯 Objectifs de la séance**
> - Appeler un LLM (local ou API) et obtenir des sorties structurées
> - Comprendre qu'un agent est une boucle autour d'un LLM (pattern ReAct)
> - Coder cette boucle à la main, puis avec LangGraph (`create_agent`)
> - Donner des outils à un agent et comprendre ce qui le fait réussir ou échouer

> **⏱️ Durée totale : 6 h**
> - Partie 1 — Notions : ~1 h
> - Partie 2 — TP guidé puis autonome : ~4 h
> - Livrable de fin de séance : ~50 min

> **📥 Livrable de fin de séance :** à déposer sur **Moodle**, dans l'espace « Livrable 1 » du cours, **à la fin de la journée**. Le sujet se trouve dans la section *Livrable 1*, en fin de notebook.

---

## Prérequis techniques

```bash
# Depuis le dossier seance1
pip install -r requirements.txt
cp .env.example .env          # puis choisir l'option A (Ollama) ou B (API personnelle)
python common/check_setup.py  # doit afficher « OK »

# Option A — LLM local avec Ollama (https://ollama.com)
ollama serve
ollama pull qwen3:8b
```

**Données utilisées :** API publique d'arXiv (articles scientifiques) et API Open-Meteo (livrable), sans clé.

**Projet fil rouge :** un assistant de veille scientifique qui cherche des articles, les lit, les résume et rédige une synthèse critique. Il s'enrichit à chaque séance.

> **Organisation des travaux pratiques**
> - Les cellules **NE PAS MODIFIER — Démonstration** sont fournies : exécutez-les et lisez le code.
> - Les cellules **✏️ À COMPLÉTER** contiennent des `TODO`.
> - L'accompagnement diminue au fil de la séance. Chaque exercice précise sa **modalité** : compléter le code fourni, implémenter à partir des étapes indiquées, réutiliser et étendre votre propre code, puis concevoir la solution en autonomie à partir des critères de réussite.

> **Indices :** pour chaque exercice, 3 niveaux d'indice sont disponibles auprès de l'enseignant (piste, démarche, morceau de code).

---
# Partie 1 — Appeler un LLM
---

### Rappels de cours

#### Ce qu'on envoie au modèle

Un appel à un LLM de type « chat » n'envoie pas une simple phrase, mais une **liste de messages**, chacun avec un **rôle** :

| Rôle | Contenu | Exemple |
| --- | --- | --- |
| `system` | Les consignes générales : rôle, ton, contraintes | « Tu es un assistant concis qui répond en français. » |
| `user` | La demande de l'utilisateur | « Qu'est-ce qu'un agent ? » |
| `assistant` | Les réponses précédentes du modèle | Permet de poursuivre une conversation |
| `tool` | Le résultat d'un outil (partie 5) | « 397.8 » |

Le modèle ne garde **aucune mémoire entre deux appels** : pour qu'il « se souvienne » d'une conversation, il faut lui renvoyer tout l'historique à chaque fois. C'est la base de la mémoire des agents (séance 2).

#### Tokens et fenêtre de contexte

Le modèle découpe le texte en **tokens** (des morceaux de mots ; en français, un mot fait en moyenne 1,3 à 1,5 token). La **fenêtre de contexte** est le nombre maximal de tokens qu'il peut traiter en une fois, entrée et sortie comprises. Tout ce qui ne tient pas dans cette fenêtre est invisible pour lui. Les API facturent au token, et un agent qui boucle consomme des tokens à chaque tour.

#### Température

Le modèle produit, à chaque pas, une **distribution de probabilités** sur le token suivant. La **température** règle la façon d'y piocher :
- **0** : on prend presque toujours le token le plus probable. Les réponses sont stables et reproductibles : c'est le réglage pour un agent (JSON, appels d'outils) ;
- **0,7 à 1** : plus de variété, utile pour la rédaction ou le brainstorming, au prix de la reproductibilité.

#### Local ou API

Tout le cours passe par un même objet `ChatOpenAI` (fichier `common/llm.py`). Ollama, comme la plupart des fournisseurs, expose une API compatible OpenAI : on change de modèle en modifiant seulement le fichier `.env`. Les petits modèles locaux sont gratuits et gardent les données sur la machine, mais ils sont moins fiables pour appeler des outils ; vous le constaterez dans l'exercice 7.

---
## Exercice 1 — Mise en place de l'environnement

> **Modalité :** suivre les étapes d'installation et vérifier que tout fonctionne.

1. `pip install -r requirements.txt`
2. Copier `.env.example` en `.env` et choisir l'option A (Ollama) ou B (API).
3. Lancer les deux cellules ci-dessous : la première doit afficher « OK ».

In [ ]:
!python common/check_setup.py

In [ ]:
# NE PAS MODIFIER — Démonstration
# Le dossier common/ se trouve dans le dossier de la séance
from common.llm import get_llm
llm = get_llm()

from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("Tu es un assistant concis qui répond en français."),
    HumanMessage("Qu'est-ce qu'un agent en IA, en une phrase ?"),
]
reponse = llm.invoke(messages)
print(reponse.content)
print(reponse.usage_metadata)  # nombre de tokens consommés

---
## Exercice 2 — Effet de la température et du prompt système

> **Modalité :** formuler des prédictions, puis les confronter aux résultats.

Avant d'exécuter la cellule suivante, **écrivez vos prédictions** : que va-t-il se passer pour chaque configuration ?

| Configuration | Ma prédiction | Ce que j'observe |
| --- | --- | --- |
| Température 0, lancée 3 fois | | |
| Température 1, lancée 3 fois | | |
| Prompt système « réponds en un mot » | | |
| Prompt système « tu es un pirate » | | |

In [ ]:
# NE PAS MODIFIER — Démonstration
question = "Donne un nom pour un assistant de veille scientifique."

for t in [0.0, 1.0]:
    llm_t = get_llm(temperature=t)
    print(f"--- température {t}")
    for _ in range(3):
        print(llm_t.invoke(question).content)

for consigne in ["Réponds en un seul mot.", "Tu es un pirate."]:
    print(f"--- système : {consigne}")
    print(llm.invoke([SystemMessage(consigne), HumanMessage(question)]).content)

---
# Partie 2 — Sorties structurées

### Rappels de cours

#### Pourquoi structurer la sortie ?

Dans un agent, la réponse du modèle n'est pas destinée à être lue, mais à être **exploitée par du code** : remplir une base, choisir une branche, appeler une fonction. Un texte libre (« L'article a été écrit par Camille Martin et deux collègues en 2025 ») est difficile à exploiter. Il faut une forme fixe :

```json
{"titre": "...", "auteurs": ["Camille Martin", "..."], "annee": 2025, "theme": "..."}
```

#### Pydantic

[Pydantic](https://docs.pydantic.dev) décrit cette forme en Python et **vérifie** qu'une donnée la respecte : types, champs obligatoires, valeurs par défaut. Une donnée invalide lève une erreur explicite au lieu de se propager silencieusement.

```python
class Article(BaseModel):
    titre: str
    annee: int          # "2025" sera converti en 2025 ; "inconnue" lèvera une erreur
```

#### `with_structured_output`

`llm.with_structured_output(Article)` fait trois choses : il envoie au modèle le **schéma** attendu, récupère sa réponse et la **valide** avec Pydantic. Vous obtenez directement un objet `Article`.

#### Point de vigilance

Un modèle à qui l'on impose un champ obligatoire **le remplira, quitte à inventer**. Si une information peut manquer, le champ doit être **optionnel** (`int | None = None`) et la consigne doit autoriser la valeur nulle. C'est l'objet de la seconde partie de l'exercice 3.

Ce mécanisme est celui qu'utilisent les **outils** : quand le modèle appelle un outil, il produit précisément un objet structuré (le nom de l'outil et ses arguments), validé par un schéma.

## Exercice 3 — Extraction d'informations structurées

> **Modalité :** compléter le code fourni.

Complétez le modèle `Article` puis extrayez les informations du résumé ci-dessous.

In [ ]:
# ✏️ À COMPLÉTER
from pydantic import BaseModel, Field

RESUME = """
Assistants de recherche autonomes : une étude empirique.
Camille Martin, Yanis Benali, Sofia Rossi (2025).
Nous évaluons des agents fondés sur des LLM capables de chercher, lire et
synthétiser la littérature scientifique. Sur 120 tâches, les agents munis
d'outils de recherche dépassent nettement les modèles seuls, mais restent
sensibles aux documents trompeurs.
"""


class Article(BaseModel):
    titre: str = Field(description="Titre de l'article")
    # TODO : ajouter les champs auteurs (liste de chaînes), annee (entier)
    #        et theme (chaîne courte décrivant le sujet)
    ...


extracteur = llm.with_structured_output(Article)
# TODO : appeler extracteur.invoke(...) sur RESUME et afficher le résultat

In [ ]:
# ✏️ À COMPLÉTER
# Que se passe-t-il si le texte ne contient pas d'année ? Testez, puis rendez
# le champ annee optionnel (int | None = None) et recommencez.

---
# Partie 3 — Qu'est-ce qu'un agent ?

### Rappels de cours

#### Du LLM à l'agent

Un LLM seul est une **fonction** : il reçoit un texte et produit un texte, en un seul passage. Il ne peut ni consulter une source, ni calculer de façon fiable, ni vérifier ce qu'il affirme. Tout ce qu'il « sait » vient de ses données d'entraînement.

Un **agent** place ce LLM dans une **boucle**. À chaque tour, le LLM ne produit plus seulement une réponse : il **décide de la prochaine action**. Le programme exécute cette action et renvoie le résultat au LLM, qui décide de la suite.

| | LLM seul | Agent |
| --- | --- | --- |
| Nombre d'appels au modèle | Un seul | Plusieurs, jusqu'à la réponse finale |
| Accès au monde extérieur | Aucun | Par des outils (recherche, calcul, fichiers, API) |
| Qui choisit les étapes ? | Personne : il n'y a qu'une étape | Le LLM, à chaque tour |
| Erreurs typiques | Hallucination | Hallucination, mauvais outil, boucle sans fin, coût qui dérive |

#### Les quatre composantes

1. **Le LLM**, qui raisonne et choisit l'action suivante.
2. **Les outils** : des fonctions écrites par le développeur (chercher un article, lire un PDF, calculer). Le LLM ne les exécute pas : il **demande** leur exécution, c'est le code qui les lance.
3. **La boucle**, qui alterne décision du LLM et exécution par le code, jusqu'à un critère d'arrêt.
4. **La mémoire** : au minimum, l'historique des messages du tour en cours, que le LLM relit à chaque étape. La mémoire entre conversations sera traitée en séance 2.

#### Le pattern ReAct

ReAct (*Reasoning + Acting*, Yao et al., 2022) structure chaque tour en trois temps :

```
        ┌──────────────────────────────────────────────┐
        ▼                                              │
  Pensée : « il me faut 17 % de 2 340 »                │
        │                                              │
  Action : calculatrice[0.17 * 2340]                   │
        │                                              │
  (le code exécute l'outil)                            │
        │                                              │
  Observation : 397.8  ────────────────────────────────┘
        │
        ▼   (quand le LLM juge qu'il a assez d'éléments)
  Réponse finale : 1 194
```

Écrire sa **pensée** avant d'agir améliore les décisions du modèle, comme le *chain-of-thought*. L'**observation** ancre ensuite le raisonnement dans un résultat réel plutôt que dans une estimation du modèle.

**Exemple de trace complète** pour « 17 % de 2 340, arrondi, multiplié par 3 » :

| Tour | Le LLM écrit | Le code renvoie |
| --- | --- | --- |
| 1 | Pensée : je calcule 17 % de 2 340. Action : `calculatrice[0.17*2340]` | Observation : 397.8 |
| 2 | Pensée : arrondi à 398, puis × 3. Action : `calculatrice[398*3]` | Observation : 1194 |
| 3 | Réponse finale : 1 194 | — (fin de la boucle) |

#### Quand la boucle s'arrête-t-elle ?

- **Arrêt normal** : le LLM produit une réponse finale au lieu d'une action.
- **Arrêt de sécurité** : un **nombre maximal d'étapes** est atteint. C'est indispensable : un modèle qui ne trouve pas ce qu'il cherche peut rappeler le même outil indéfiniment, et chaque tour consomme des tokens.
- **Arrêt sur erreur** : à éviter. Un outil qui échoue doit renvoyer un message d'erreur lisible, pour que le LLM puisse corriger son action.

#### Agent ou simple programme ?

Si les étapes sont toujours les mêmes et dans le même ordre, un **programme classique** (un *workflow*) suffit : il est plus rapide, moins cher et plus prévisible. L'agent se justifie quand **le chemin dépend de ce qu'on découvre en route**. Ce choix sera approfondi en séance 3.

#### Vue d'ensemble : ce que le cours va construire autour de cette boucle

Chaque notion du cours répond à une limite de la boucle de base :

| Notion | Problème qu'elle résout | Où |
| --- | --- | --- |
| **Tool calling** | Analyser du texte libre (« Action : … ») est fragile : le modèle produit directement un appel structuré | Séance 1, parties 4 et 5 |
| **Outils** (`@tool`) | Donner au modèle un accès fiable au monde : recherche, lecture, calcul | Séance 1, partie 5 |
| **LangGraph** | Ne plus réécrire la boucle à la main, et pouvoir la structurer en graphe | Séance 1 (partie 4), puis séance 2 |
| **Mémoire** | Se souvenir d'une conversation à l'autre, sans saturer la fenêtre de contexte | Séance 2 |
| **RAG agentique** | Interroger un corpus trop grand pour la fenêtre de contexte, en laissant l'agent décider quand chercher | Séance 2 |
| **Graphes et patterns** | Combiner étapes fixes et décisions du LLM (routage, critique, parallélisation) | Séances 2 et 3 |
| **Validation humaine** | Empêcher une action sensible sans accord explicite | Séance 3 |
| **MCP** | Écrire un outil une fois et le réutiliser dans n'importe quel agent ou application | Séance 3 |
| **Multi-agents** | Séparer des rôles qui ont des consignes et des contextes différents | Séance 3 |
| **Évaluation et sécurité** | Mesurer la qualité d'un agent et le protéger contre les manipulations | Séance 4 |

> **À retenir :** un agent, c'est une boucle `while` autour d'un LLM. Le LLM choisit, le code exécute, et une limite d'étapes garantit que la boucle s'arrête. L'exercice 4 vous fait écrire cette boucle vous-même.

## Exercice 4 — Reconstitution de la boucle ReAct

> **Modalité :** compléter le code fourni.

L'outil `calculatrice` et le prompt système sont fournis. Les lignes de la boucle ReAct sont données **dans le désordre** : remettez-les dans l'ordre (et à la bonne indentation) dans la fonction `agent_maison`.

```text
messages.append(HumanMessage(f"Observation : {observation}"))
for etape in range(max_etapes):
return "Nombre maximal d'étapes atteint."
    return texte.split("Réponse finale :")[-1].strip()
texte = llm.invoke(messages).content
if "Réponse finale :" in texte:
action = re.search(r"Action : (\w+)\[(.*)\]", texte)
messages.append(AIMessage(texte))
observation = OUTILS[nom](argument)
nom, argument = action.groups()
if action:
print(texte)
```

In [ ]:
# ✏️ À COMPLÉTER
import re
from langchain_core.messages import AIMessage


def calculatrice(expression: str) -> str:
    """Évalue une expression arithmétique simple."""
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Erreur : expression non autorisée."
    try:
        return str(eval(expression))  # sûr ici : seuls des chiffres et opérateurs passent
    except Exception as e:
        return f"Erreur : {e}"


OUTILS = {"calculatrice": calculatrice}

SYSTEME_REACT = """Tu résous la question pas à pas.
À chaque étape, écris :
Pensée : ton raisonnement
Action : calculatrice[expression]
Puis attends l'observation. Quand tu as la réponse, écris :
Réponse finale : ta réponse"""


def agent_maison(question: str, max_etapes: int = 5) -> str:
    messages = [SystemMessage(SYSTEME_REACT), HumanMessage(question)]
    # TODO : remettre ici les lignes dans le bon ordre
    ...


print(agent_maison("Combien font 17 % de 2 340, arrondi à l'unité, multiplié par 3 ?"))

---
# Partie 4 — La boucle d'agent avec LangGraph

### Rappels de cours

#### La limite de la boucle écrite à la main

Dans l'exercice 4, le modèle écrit son action en texte libre (`Action : calculatrice[...]`) et une expression régulière la retrouve. C'est fragile : un espace en trop, une autre langue, une parenthèse mal fermée, et l'action est perdue.

#### Le tool calling natif

Les modèles récents savent produire un **appel d'outil structuré**. On leur transmet la liste des outils avec leur schéma ; quand ils veulent en utiliser un, ils ne renvoient pas du texte mais un objet de ce type :

```json
{"name": "calculatrice_tool", "args": {"expression": "0.17*2340"}, "id": "call_01"}
```

Le code exécute l'outil et renvoie le résultat dans un message de rôle `tool`, rattaché à l'appel par son `id`. Plus besoin d'expression régulière : c'est le même mécanisme de sortie structurée que dans la partie 2.

#### LangGraph et `create_agent`

[LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) représente un agent comme un **graphe** : des **nœuds** (des fonctions) reliés par des **arêtes**, qui se partagent un **état** (ici, la liste des messages). `create_agent` construit pour vous le graphe de la boucle ReAct :

```
START → modèle ──(appel d'outil ?)── oui ──→ outils ──→ modèle
                        └── non ──→ END
```

| Boucle de l'exercice 4 | Équivalent dans LangGraph |
| --- | --- |
| `messages` | L'**état** du graphe |
| `llm.invoke(messages)` | Le nœud **modèle** |
| `OUTILS[nom](argument)` | Le nœud **outils** |
| `if "Réponse finale"` | L'arête conditionnelle : pas d'appel d'outil → `END` |
| `max_etapes` | `recursion_limit` dans la configuration |

Pourquoi un framework si la boucle tient en quelques lignes ? Parce qu'il apporte ce que vous auriez à réécrire ensuite : la sauvegarde de l'état (mémoire, séance 2), la reprise après une pause (validation humaine, séance 3), l'affichage des étapes en direct, et la possibilité de remplacer la boucle par un graphe sur mesure (séance 2).

> Depuis LangChain v1, `create_agent` (module `langchain.agents`) remplace l'ancien `create_react_agent` de LangGraph.

## Exercice 5 — La boucle ReAct avec LangGraph

> **Modalité :** compléter le code fourni.

1. Écrivez la docstring de l'outil (c'est ce que lira le modèle).
2. Donnez l'outil à l'agent et posez la même question qu'à l'exercice 4.
3. Dans la cellule suivante, **reliez chaque message de la trace LangGraph à une étape de votre boucle maison**.

In [ ]:
# ✏️ À COMPLÉTER
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def calculatrice_tool(expression: str) -> str:
    """TODO : décrire l'outil et donner un exemple d'expression."""
    return calculatrice(expression)  # réutilise votre fonction de l'exercice 4


agent = create_agent(
    model=llm,
    tools=[...],          # TODO
    system_prompt="...",  # TODO : demander d'utiliser la calculatrice pour tout calcul
)

resultat = agent.invoke(
    {"messages": [("user", ...)]},  # TODO : la question de l'exercice 4
    config={"recursion_limit": 10},
)
for m in resultat["messages"]:
    m.pretty_print()

*Votre correspondance trace LangGraph ↔ boucle maison :*

- `HumanMessage` → …
- `AIMessage` avec `tool_calls` → …
- `ToolMessage` → …
- dernier `AIMessage` → …

---
# Partie 5 — Les outils

### Rappels de cours

#### Qu'est-ce qu'un outil ?

Un outil est une **fonction Python ordinaire** que l'on rend utilisable par le modèle. Le décorateur `@tool` en extrait un **schéma** :

| Élément de la fonction | Ce que le modèle reçoit |
| --- | --- |
| Le nom (`search_arxiv`) | Le nom de l'outil |
| La docstring | La **description** : quand et comment l'utiliser |
| Les paramètres typés (`query: str`) | Le schéma JSON des arguments |

Le modèle ne voit **jamais le code** de l'outil : il ne connaît que ce schéma. Une fonction parfaite mais mal décrite sera mal utilisée.

#### La docstring est un prompt

C'est à partir de la description que le modèle décide **s'il** appelle l'outil, **lequel** choisir parmi plusieurs, et **comment** remplir les arguments. Une bonne description précise :
- ce que fait l'outil et **quand** l'utiliser ;
- le **format** attendu des arguments (« mots-clés en anglais », « date au format AAAA-MM-JJ ») ;
- ce que l'outil **renvoie**, y compris en cas d'échec.

#### Robustesse

- **Valider les arguments** côté code : le modèle peut proposer n'importe quelle valeur (`max_results=500`). On les borne.
- **Ne pas lever d'exception** : un outil qui plante arrête l'agent. Un outil qui renvoie « Erreur : article introuvable » permet au modèle de reformuler ou de prévenir l'utilisateur.
- **Limiter la taille des résultats** : un PDF entier peut saturer la fenêtre de contexte ; on tronque.

#### Sécurité

Un outil donne au modèle un **pouvoir d'action**. Plus il est puissant (exécuter du code, écrire un fichier, envoyer un mail), plus il faut le restreindre :
- **bac à sable** pour l'exécution de code (exercice 9) ;
- **moindre privilège** : ne donner à l'agent que les outils nécessaires à sa tâche ;
- **validation humaine** avant les actions irréversibles (séance 3).

La séance 4 montrera comment un document piégé peut détourner un agent qui dispose d'outils trop permissifs.

#### Et ensuite ?

Les outils de cette séance sont définis dans le notebook, donc liés à cet agent. En séance 3, vous les servirez par un **serveur MCP** pour qu'un même outil soit utilisable par n'importe quel agent ou application.

## Exercice 6 — Analyse d'un outil existant

> **Modalité :** exécuter le code fourni et analyser les résultats.

Étudiez l'outil ci-dessous puis répondez aux 3 questions.

In [ ]:
# NE PAS MODIFIER — Démonstration
import arxiv


@tool
def search_arxiv(query: str, max_results: int = 5) -> list[dict]:
    """Cherche des articles scientifiques sur arXiv.

    Args:
        query: mots-clés en anglais, par exemple "LLM agents tool use".
        max_results: nombre d'articles à renvoyer (1 à 10).

    Returns:
        Une liste d'articles avec titre, auteurs, date, résumé et lien PDF.
    """
    max_results = max(1, min(max_results, 10))
    try:
        recherche = arxiv.Search(query=query, max_results=max_results,
                                 sort_by=arxiv.SortCriterion.SubmittedDate)
        return [
            {
                "titre": r.title,
                "auteurs": [a.name for a in r.authors][:5],
                "date": r.published.date().isoformat(),
                "resume": r.summary[:800],
                "pdf_url": r.pdf_url,
            }
            for r in arxiv.Client().results(recherche)
        ]
    except Exception as e:
        return [{"erreur": f"Recherche impossible : {e}"}]


search_arxiv.invoke({"query": "LLM agents tool use", "max_results": 2})

**Questions**
1. Pourquoi la docstring précise-t-elle que les mots-clés sont « en anglais » ?
2. À quoi sert la ligne `max_results = max(1, min(max_results, 10))` ?
3. Pourquoi renvoyer `[{"erreur": ...}]` plutôt que de laisser l'exception remonter ?

*Vos réponses :*

## Exercice 7 — Diagnostic d'outils défaillants

> **Modalité :** diagnostiquer et corriger le code fourni.

L'agent ci-dessous échoue sur les 4 tâches. Les causes sont dans la définition des outils. Lisez les traces, trouvez chaque cause et corrigez-la.

In [ ]:
# NE PAS MODIFIER — Démonstration
@tool
def outil1(x):
    """Fait un truc."""
    return calculatrice(x)


@tool
def date_du_jour() -> str:
    """Donne la date."""
    import datetime
    return datetime.date.today().strftime("%d/%m")


@tool
def convertir(valeur: float, unite: str) -> str:
    """Convertit des kilomètres en miles."""
    if unite == "km":
        return str(valeur * 0.621)
    raise ValueError("unité inconnue")


agent_bugge = create_agent(model=llm, tools=[outil1, date_du_jour, convertir])

TACHES = [
    "Combien font 1234 * 5678 ?",
    "En quelle année sommes-nous ?",
    "Convertis 42 miles en kilomètres.",
    "Combien de jours reste-t-il avant le 31 décembre de cette année ?",
]
for t in TACHES:
    print("=" * 60, "\n", t)
    try:
        r = agent_bugge.invoke({"messages": [("user", t)]}, config={"recursion_limit": 8})
        print(r["messages"][-1].content)
    except Exception as e:
        print("ÉCHEC :", e)

In [ ]:
# ✏️ À COMPLÉTER
# Vos outils corrigés et le nouvel agent

*Pour chaque tâche : cause de l'échec → correction apportée.*

1. …
2. …
3. …
4. …

---
# Travail en autonomie

Les exercices suivants s'appuient sur le code que vous avez déjà écrit.

## Exercice 8 — Outil de lecture de PDF

> **Modalité :** implémenter à partir des étapes indiquées.

Les étapes sont données ; écrivez le code et la docstring. L'outil doit gérer un lien cassé ou un PDF illisible.

In [ ]:
# ✏️ À COMPLÉTER
import io
import requests
from pypdf import PdfReader


@tool
def read_pdf(url: str, max_chars: int = 8000) -> str:
    """TODO : docstring sur le modèle de search_arxiv (rôle, arguments, retour)."""
    # 1. télécharger le PDF avec requests.get (penser au timeout)
    # 2. extraire le texte page par page avec PdfReader(io.BytesIO(...))
    # 3. tronquer à max_chars
    # 4. en cas d'échec, renvoyer une erreur lisible au lieu de planter
    ...

## Exercice 9 — Outil d'exécution de code isolé

> **Modalité :** réutiliser et étendre votre propre code.

En vous inspirant de vos deux outils précédents, écrivez `run_python(code: str) -> str` qui exécute du code dans un **sous-processus**, avec un délai maximal de 10 s et **sans accès réseau**. Seul le nom de l'outil est imposé.

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 9 — run_python

## Exercice 10 — Influence des descriptions d'outils

> **Modalité :** réutiliser et étendre votre propre code.

Reprenez vos outils `search_arxiv`, `read_pdf` et `run_python`. Lancez 5 tâches de votre choix avec une version **floue** puis **précise** de leurs descriptions.

| Tâche | Description floue : réussite ? | Description précise : réussite ? | Observation |
| --- | --- | --- | --- |
| | | | |

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 10 — votre expérience

## Exercice 11 — Projet fil rouge

> **Modalité :** concevoir la solution en autonomie, à partir des critères de réussite.

Assemblez ce que vous avez écrit (l'agent de l'exercice 5, `search_arxiv`, `read_pdf`) en un agent de veille : il reçoit un thème, trouve 5 articles récents et rédige pour chacun un résumé de 3 lignes. Créez votre dépôt Git et commitez ce premier agent.

**Critères :** les 5 résumés citent leur article, l'agent s'arrête seul, aucune erreur d'outil ne le fait planter.

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 11 — premier agent de veille

---
# Livrable 1 — Agent météo

> **📥 Dépôt sur Moodle — à la fin de la journée**<br>
> Déposez ce notebook exécuté (`S1_agent_outille.ipynb`) avec les traces de l'agent dans l'espace **« Livrable 1 »** du cours sur Moodle.


**Sujet.** Créez avec `create_agent` un agent qui répond aux 5 questions ci-dessous grâce à l'API **Open-Meteo** (gratuite, sans clé).

- Géocodage : `https://geocoding-api.open-meteo.com/v1/search?name=Paris&count=1`
- Prévisions : `https://api.open-meteo.com/v1/forecast?latitude=48.85&longitude=2.35&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=Europe/Paris`

Écrivez **2 outils** : l'un récupère une prévision pour une ville, l'autre calcule une statistique (moyenne, écart, maximum).

**Critères de réussite :** le notebook s'exécute de bout en bout, les outils sont typés et décrits, une erreur d'API (ville inconnue) est gérée, au moins 4 réponses sur 5 sont justes.

In [ ]:
# ✏️ À COMPLÉTER
QUESTIONS_LIVRABLE = [
    "Quelle sera la température maximale moyenne à Saint-Denis sur les 7 prochains jours ?",
    "Quel jour de la semaine sera le plus chaud à Lyon ?",
    "Quel est l'écart entre le maximum et le minimum de température à Marseille demain ?",
    "Va-t-il pleuvoir plus à Brest ou à Nice cette semaine ?",
    "Quelle est la météo prévue à Xyzville ?",  # ville inconnue : l'agent doit le dire
]

# Vos outils et votre agent ici

**Questions de compréhension** (3 lignes chacune)

1. À quoi sert la docstring d'un outil ?
2. Que fait `recursion_limit` et pourquoi est-il indispensable ?
3. Quelle différence entre votre boucle maison et `create_agent` ?

*Vos réponses :*

---
# Exercices d'approfondissement

Destinés aux étudiants ayant terminé ; non évalués, ils peuvent être poursuivis en dehors de la séance.

1. Affichez les étapes de l'agent en direct avec `agent.stream(..., stream_mode="updates")`.
2. Ajoutez un outil Semantic Scholar et laissez l'agent choisir entre arXiv et Semantic Scholar.
3. Comparez deux modèles (deux locaux, ou un local et une API) sur les 5 tâches de l'expérience contrôlée.
4. Réécrivez `search_arxiv` en version asynchrone et lancez deux recherches en parallèle.

In [ ]:
# ✏️ À COMPLÉTER
# Exercices d'approfondissement

---
## Journal de bord

- Un échec observé aujourd'hui :
- Sa cause :
- Ce que j'ai fait pour le corriger :